# Mission Control AI

Sistema inteligente de monitoramento de uma missão espacial com Python e Llama via Ollama.

## 1. Instalar o Ollama e o modelo

Execute esta célula e aguarde o download do modelo `llama3.2:3b`.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama -q

import subprocess
import time

servidor = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
!ollama pull llama3.2:3b
print("Ollama e Llama instalados com sucesso!")

## 2. Importar bibliotecas e definir o contexto da IA

In [ ]:
import random
import ollama

SYSTEM_PROMPT = """
Você é a inteligência artificial do Mission Control de uma missão espacial
experimental e fictícia criada para um trabalho acadêmico. Analise os dados
simulados de temperatura, energia, comunicação e status do módulo.
Analise a telemetria e os alertas fornecidos. Sua única tarefa é classificar
a situação pelas regras: se todos os parâmetros estiverem normais, responda
NORMAL; se houver temperatura crítica, energia crítica, comunicação perdida
ou falha no módulo, responda CRÍTICO; nos demais alertas, responda ATENÇÃO.
Responda com apenas uma dessas três palavras, sem explicações.
"""

print("Contexto da IA definido.")

## 3. Criar a lógica da missão

In [ ]:
def gerar_dados():
    """Gera dados aleatórios de uma missão espacial fictícia."""
    return {
        "temperatura": random.randint(20, 100),
        "energia": random.randint(10, 100),
        "comunicacao": random.choice(["estável", "instável", "perdida"]),
        "modulo": random.choice(["operacional", "atenção", "falha"]),
    }


def gerar_alertas(dados):
    """Aplica regras e retorna alertas e respostas automáticas."""
    alertas = []
    acoes = []
    critico = False

    if dados["temperatura"] > 80:
        critico = True
        alertas.append("Temperatura crítica")
        acoes.append("Ativar resfriamento de emergência")
    elif dados["temperatura"] > 60:
        alertas.append("Temperatura elevada")

    if dados["energia"] < 20:
        critico = True
        alertas.append("Energia crítica")
        acoes.append("Ativar modo de economia")
    elif dados["energia"] < 40:
        alertas.append("Energia baixa")

    if dados["comunicacao"] == "instável":
        alertas.append("Comunicação instável")
        acoes.append("Reiniciar antena principal")
    elif dados["comunicacao"] == "perdida":
        critico = True
        alertas.append("Comunicação perdida")
        acoes.append("Ativar canal de comunicação reserva")

    if dados["modulo"] == "atenção":
        alertas.append("Módulo requer atenção")
    elif dados["modulo"] == "falha":
        critico = True
        alertas.append("Falha no módulo")
        acoes.append("Isolar módulo e solicitar diagnóstico")

    if not alertas:
        alertas.append("Todos os parâmetros estão normais")
        acoes.append("Manter monitoramento")

    if critico:
        nivel = "CRÍTICO"
    elif alertas[0] == "Todos os parâmetros estão normais":
        nivel = "NORMAL"
    else:
        nivel = "ATENÇÃO"

    return nivel, alertas, acoes


def analisar_com_ia(dados, alertas):
    """Envia a telemetria ao Llama e retorna a classificação da IA."""
    mensagem = f"""
    Temperatura: {dados['temperatura']} °C
    Energia: {dados['energia']}%
    Comunicação: {dados['comunicacao']}
    Status do módulo: {dados['modulo']}
    Alertas automáticos: {', '.join(alertas)}
    Classifique a situação usando somente NORMAL, ATENÇÃO ou CRÍTICO.
    """

    resposta = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": mensagem},
        ],
        options={"temperature": 0.0, "num_predict": 10},
    )
    texto = resposta["message"]["content"].strip().upper()

    for nivel in ["CRÍTICO", "ATENÇÃO", "NORMAL"]:
        if nivel in texto:
            return nivel

    raise ValueError(f"Resposta inesperada da IA: {texto}")


def exibir_status(nome, dados, nivel, alertas, acoes, classificacao_ia):
    """Exibe os dados da missão de forma organizada."""
    print("=" * 60)
    print(f"MISSION CONTROL AI - {nome.upper()}")
    print("=" * 60)
    print(f"Temperatura : {dados['temperatura']} °C")
    print(f"Energia     : {dados['energia']}%")
    print(f"Comunicação : {dados['comunicacao']}")
    print(f"Módulo      : {dados['modulo']}")
    print(f"Classificação: {nivel}")
    print("-" * 60)
    print("ALERTAS AUTOMÁTICOS")
    for alerta in alertas:
        print(f"- {alerta}")
    print("-" * 60)
    print("RESPOSTAS AUTOMÁTICAS")
    for acao in acoes:
        print(f"- {acao}")
    print("-" * 60)
    print("ANÁLISE DA IA - LLAMA 3.2 3B")
    print(f"Classificação da IA: {classificacao_ia}")
    print("=" * 60)


def executar_missao(nome, dados):
    nivel, alertas, acoes = gerar_alertas(dados)
    classificacao_ia = analisar_com_ia(dados, alertas)
    exibir_status(nome, dados, nivel, alertas, acoes, classificacao_ia)

print("Funções do Mission Control carregadas.")

## 4. Cenário normal

In [ ]:
cenario_normal = {
    "temperatura": 28,
    "energia": 85,
    "comunicacao": "estável",
    "modulo": "operacional",
}

executar_missao("Cenário normal", cenario_normal)

## 5. Situação crítica simulada

In [ ]:
cenario_critico = {
    "temperatura": 95,
    "energia": 18,
    "comunicacao": "instável",
    "modulo": "falha",
}

executar_missao("Situação crítica", cenario_critico)

## 6. Dados aleatórios

In [ ]:
dados_aleatorios = gerar_dados()
executar_missao("Dados aleatórios", dados_aleatorios)